In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
DATA = Path("data")

## An image is a grid of numbers

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
digit = x_train[np.where(y_train == 9)[0][0]]          # the first 9 of the training set
print(digit.shape, digit.dtype, digit.min(), digit.max())
print(digit[4:14, 10:20])                              # a 10 x 10 corner of it, as numbers
pd.DataFrame(digit).to_csv(DATA / "digit9.csv", index=False)

## Weights of the first Dense layer: pixels x nodes

In [ ]:
for h, w, nodes in ((28, 28, 128), (40, 40, 100), (1000, 1000, 500)):
    print(f"{h} x {w} image, {nodes} nodes: {h * w * nodes:,} weights")

## Flattening moves neighbours apart
Pixel (row r, column c) goes to position 28 r + c of the flattened row.

In [ ]:
flat = digit.reshape(-1)
r, c = 10, 14
print("pixel (10, 14) ->", 28 * r + c, " pixel below it (11, 14) ->", 28 * (r + 1) + c)
print("same values:", flat[28 * r + c] == digit[r, c], flat[28 * (r + 1) + c] == digit[r + 1, c])

## The same ANN on normal and on scrambled images
One random permutation of the 784 pixel positions, applied to every training and test image.
ANN as in the MNIST ANN Note: Flatten, Dense(128, relu), Dense(10, softmax); 5 epochs; 3 seeds.

In [ ]:
perm = np.random.default_rng(0).permutation(784)
def scramble(x):
    return x.reshape(len(x), -1)[:, perm].reshape(x.shape)

xs_train, xs_test = scramble(x_train), scramble(x_test)
np.savez_compressed(DATA / "scrambled_examples.npz", normal=x_test[:4], scrambled=xs_test[:4], labels=y_test[:4])

def ann():
    return keras.Sequential([keras.Input(shape=(28, 28)), keras.layers.Flatten(),
                             keras.layers.Dense(128, activation="relu"), keras.layers.Dense(10, activation="softmax")])

rows = []
for name, (xtr, xte) in (("normal", (x_train, x_test)), ("scrambled", (xs_train, xs_test))):
    for seed in (1, 2, 3):
        keras.utils.set_random_seed(seed)
        m = ann()
        m.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
        m.fit(xtr / 255.0, y_train, epochs=5, batch_size=128, verbose=0)
        rows.append(dict(images=name, seed=seed, test_acc=m.evaluate(xte / 255.0, y_test, verbose=0)[1]))
res = pd.DataFrame(rows)
res.to_csv(DATA / "scramble_results.csv", index=False)
print(res.groupby("images").test_acc.agg(["mean", "min", "max"]).round(4))
print("parameters:", ann().count_params())